# Tabelas no catálogo `glue` no AWS Glue 6.0

Este notebook é um **teste**: ele lê o CSV pequeno da `landing` e cria **tabelas registradas no catálogo `glue`** (banco `silver`) nos formatos Iceberg, Delta Lake, Hudi, Parquet, CSV e ORC. A Parte A usa **Spark SQL** e cria, em cada formato, uma tabela **gerenciada** (sem `LOCATION`) e uma **externa** (com `LOCATION`). A Parte B usa **DynamicFrame** (`getSink`, `from_catalog` e `write_dynamic_frame.from_catalog`) nos formatos que ele suporta. Todas as células podem ser reexecutadas e, se alguma falhar, a falha é um resultado válido do teste: ela também aconteceria na AWS.

## 1. Sessão

O runtime **AWS Glue 6.0** já configura o catálogo, as extensões e o acesso ao `s3`; só precisamos obter a sessão a partir do `GlueContext`.

In [ ]:
from awsglue.context import GlueContext
from awsglue.dynamicframe import DynamicFrame
from pyspark.context import SparkContext

glueContext = GlueContext(SparkContext.getOrCreate())
spark = glueContext.spark_session

## 2. Funções de apoio

`limpar` apaga uma pasta pela API de sistema de arquivos do Hadoop (a mesma que o Spark usa para o `s3://`) e `existe` verifica se ela existe. Usamos `limpar` antes de recriar uma tabela **externa**: o `DROP TABLE` de uma externa remove só a entrada do catálogo e deixa os arquivos, que seriam lidos de novo na recriação. Nas **gerenciadas** `tg_*`, o `DROP TABLE` apaga entrada e arquivos, mas chamamos `limpar` logo depois para remover a pasta inteira (restos de execuções anteriores ou arquivos que o `DROP` possa deixar), já que sem isso a recriação poderia ler essas sobras.

In [ ]:
def _fs(caminho):
    hadoop_path = spark._jvm.org.apache.hadoop.fs.Path(caminho)
    return hadoop_path, hadoop_path.getFileSystem(spark._jsc.hadoopConfiguration())


def limpar(caminho):
    hadoop_path, fs = _fs(caminho)
    return fs.delete(hadoop_path, True)


def existe(caminho):
    hadoop_path, fs = _fs(caminho)
    return fs.exists(hadoop_path)

## 3. Preparar a origem

O CSV **com cabeçalho** fica em `landing/teste_glue/pessoas/` e é a mesma origem de outros notebooks de teste. Se ele ainda não existir (este notebook foi executado primeiro), nós o criamos aqui com o mesmo conteúdo; se já existir, nada muda.

In [ ]:
ORIGEM = "s3://notislake/landing/teste_glue/pessoas/"
schema = "id INT, nome STRING, area STRING, valor DOUBLE, dt STRING"

if not existe(ORIGEM):
    pessoas = spark.createDataFrame(
        [
            (1, "Ana", "Dados", 100.5, "2026-01-01"),
            (2, "Bruno", "Plataforma", 200.0, "2026-01-02"),
            (3, "Carla", "Segurança", 300.25, "2026-01-03"),
            (4, "Diego", "Dados", 400.0, "2026-01-04"),
            (5, "Elisa", "Plataforma", 500.75, "2026-01-05"),
        ],
        schema,
    )
    pessoas.coalesce(1).write.mode("overwrite").option("header", True).csv(ORIGEM)

print("origem existe:", existe(ORIGEM))

## 4. Ler o CSV

Lemos o CSV com **schema explícito** e criamos a view temporária `pessoas_csv`, usada como fonte de todas as tabelas (`CREATE TABLE ... AS SELECT`).

In [ ]:
df = spark.read.schema(schema).option("header", True).csv(ORIGEM)
df.createOrReplaceTempView("pessoas_csv")
df.orderBy("id").show()
print(df.count())

# Parte A. Spark SQL

Para cada formato criamos duas tabelas no banco `silver`:

- **gerenciada** `tg_<formato>`: `CREATE TABLE` **sem** `LOCATION`. O catálogo escolhe a pasta, `s3://notislake/silver/tg_<formato>`, e o `DROP TABLE` apaga os arquivos (o código ainda chama `limpar` em seguida, para a reexecução não ler sobras de execuções anteriores);
- **externa** `te_<formato>`: `CREATE TABLE` **com** `LOCATION 's3://notislake/silver/te_<formato>/'`. O `DROP TABLE` remove só a entrada do catálogo.

Cada tabela é mostrada com `DESCRIBE TABLE EXTENDED` (as linhas `Type` e `Location` dizem se ela é gerenciada ou externa; a exceção é o Iceberg, explicada na seção dele) e com um `SELECT`. Cada bloco começa com `DROP TABLE IF EXISTS` para poder ser reexecutado. O estado esperado ao final de cada formato é de **5** linhas.

## 5. Iceberg

O Iceberg só funciona pelo catálogo nomeado `glue_catalog` (`glue_catalog.<banco>.<tabela>`); com nome simples, `USING iceberg` falha. **No Glue o Iceberg é sempre EXTERNAL**, mesmo sem `LOCATION`: nesse caso o local padrão sai do `LocationUri` do banco `silver`, e o `DROP TABLE` não apaga os arquivos. O `DESCRIBE TABLE EXTENDED` do Spark rotula toda tabela Iceberg como `MANAGED`, mas o catálogo `glue` a registra como `EXTERNAL_TABLE`; o resumo no fim do notebook lê o tipo direto do catálogo. O `DROP TABLE ... PURGE` do código remove a entrada do catálogo e pede ao Iceberg que apague também os dados e metadados que ele conhece; mesmo assim chamamos `limpar` em seguida para remover a pasta inteira (arquivos órfãos, restos de execuções anteriores), já que sem isso a recriação poderia ler esses restos.

### Iceberg: tabela gerenciada (sem `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS glue_catalog.silver.tg_iceberg PURGE")
limpar("s3://notislake/silver/tg_iceberg/")
spark.sql("""
CREATE TABLE glue_catalog.silver.tg_iceberg
USING iceberg
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED glue_catalog.silver.tg_iceberg").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM glue_catalog.silver.tg_iceberg ORDER BY id").show()

### Iceberg: tabela externa (com `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS glue_catalog.silver.te_iceberg PURGE")
limpar("s3://notislake/silver/te_iceberg/")
spark.sql("""
CREATE TABLE glue_catalog.silver.te_iceberg
USING iceberg
LOCATION 's3://notislake/silver/te_iceberg/'
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED glue_catalog.silver.te_iceberg").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM glue_catalog.silver.te_iceberg ORDER BY id").show()

### Iceberg: INSERT, UPDATE e DELETE

Em Iceberg o Spark SQL aceita DML linha a linha. Em cada tabela inserimos o `id` 6, atualizamos o `valor` do `id` 1 e removemos o `id` 2; ao final restam **5** linhas (`1`, `3`, `4`, `5` e `6`).

In [ ]:
for tabela in ("glue_catalog.silver.tg_iceberg", "glue_catalog.silver.te_iceberg"):
    spark.sql(f"INSERT INTO {tabela} VALUES (6, 'Fabio', 'Dados', 600.0, '2026-01-06')")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

In [ ]:
for tabela in ("glue_catalog.silver.tg_iceberg", "glue_catalog.silver.te_iceberg"):
    spark.sql(f"UPDATE {tabela} SET valor = 999.0 WHERE id = 1")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

In [ ]:
for tabela in ("glue_catalog.silver.tg_iceberg", "glue_catalog.silver.te_iceberg"):
    spark.sql(f"DELETE FROM {tabela} WHERE id = 2")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

## 6. Delta Lake

O Delta Lake usa o nome simples (`silver.<tabela>`) no catálogo `glue`.

### Delta Lake: tabela gerenciada (sem `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS silver.tg_delta")
limpar("s3://notislake/silver/tg_delta/")
spark.sql("""
CREATE TABLE silver.tg_delta
USING delta
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.tg_delta").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.tg_delta ORDER BY id").show()

### Delta Lake: tabela externa (com `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS silver.te_delta")
limpar("s3://notislake/silver/te_delta/")
spark.sql("""
CREATE TABLE silver.te_delta
USING delta
LOCATION 's3://notislake/silver/te_delta/'
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.te_delta").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.te_delta ORDER BY id").show()

### Delta Lake: INSERT, UPDATE e DELETE

Em Delta Lake o Spark SQL aceita DML linha a linha. Em cada tabela inserimos o `id` 6, atualizamos o `valor` do `id` 1 e removemos o `id` 2; ao final restam **5** linhas (`1`, `3`, `4`, `5` e `6`).

In [ ]:
for tabela in ("silver.tg_delta", "silver.te_delta"):
    spark.sql(f"INSERT INTO {tabela} VALUES (6, 'Fabio', 'Dados', 600.0, '2026-01-06')")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

In [ ]:
for tabela in ("silver.tg_delta", "silver.te_delta"):
    spark.sql(f"UPDATE {tabela} SET valor = 999.0 WHERE id = 1")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

In [ ]:
for tabela in ("silver.tg_delta", "silver.te_delta"):
    spark.sql(f"DELETE FROM {tabela} WHERE id = 2")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

## 7. Hudi

O Hudi exige a chave primária (`primaryKey`) e o campo de combinação (`preCombineField`) nas propriedades da tabela; o `SELECT` lista as colunas porque o Hudi acrescenta colunas de controle (`_hoodie_*`).

### Hudi: tabela gerenciada (sem `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS silver.tg_hudi")
limpar("s3://notislake/silver/tg_hudi/")
spark.sql("""
CREATE TABLE silver.tg_hudi
USING hudi
TBLPROPERTIES (type = 'cow', primaryKey = 'id', preCombineField = 'dt')
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.tg_hudi").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.tg_hudi ORDER BY id").show()

### Hudi: tabela externa (com `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS silver.te_hudi")
limpar("s3://notislake/silver/te_hudi/")
spark.sql("""
CREATE TABLE silver.te_hudi
USING hudi
TBLPROPERTIES (type = 'cow', primaryKey = 'id', preCombineField = 'dt')
LOCATION 's3://notislake/silver/te_hudi/'
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.te_hudi").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.te_hudi ORDER BY id").show()

### Hudi: INSERT, UPDATE e DELETE

Em Hudi o Spark SQL aceita DML linha a linha. Em cada tabela inserimos o `id` 6, atualizamos o `valor` do `id` 1 e removemos o `id` 2; ao final restam **5** linhas (`1`, `3`, `4`, `5` e `6`).

In [ ]:
for tabela in ("silver.tg_hudi", "silver.te_hudi"):
    spark.sql(f"INSERT INTO {tabela} VALUES (6, 'Fabio', 'Dados', 600.0, '2026-01-06')")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

In [ ]:
for tabela in ("silver.tg_hudi", "silver.te_hudi"):
    spark.sql(f"UPDATE {tabela} SET valor = 999.0 WHERE id = 1")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

In [ ]:
for tabela in ("silver.tg_hudi", "silver.te_hudi"):
    spark.sql(f"DELETE FROM {tabela} WHERE id = 2")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

## 8. Parquet

O Parquet usa o nome simples (`silver.<tabela>`) no catálogo `glue`. Nesse formato o Spark SQL aceita `INSERT INTO` e `INSERT OVERWRITE`, mas **não** `UPDATE` nem `DELETE`.

### Parquet: tabela gerenciada (sem `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS silver.tg_parquet")
limpar("s3://notislake/silver/tg_parquet/")
spark.sql("""
CREATE TABLE silver.tg_parquet
USING parquet
TBLPROPERTIES ('classification' = 'parquet')
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.tg_parquet").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.tg_parquet ORDER BY id").show()

### Parquet: tabela externa (com `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS silver.te_parquet")
limpar("s3://notislake/silver/te_parquet/")
spark.sql("""
CREATE TABLE silver.te_parquet
USING parquet
LOCATION 's3://notislake/silver/te_parquet/'
TBLPROPERTIES ('classification' = 'parquet')
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.te_parquet").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.te_parquet ORDER BY id").show()

### Parquet: INSERT INTO e INSERT OVERWRITE

`INSERT INTO` acrescenta o `id` 6 (a tabela passa a ter 6 linhas); `INSERT OVERWRITE` troca todo o conteúdo pelo CSV de origem (volta para **5** linhas).

In [ ]:
for tabela in ("silver.tg_parquet", "silver.te_parquet"):
    spark.sql(f"INSERT INTO {tabela} VALUES (6, 'Fabio', 'Dados', 600.0, '2026-01-06')")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

In [ ]:
for tabela in ("silver.tg_parquet", "silver.te_parquet"):
    spark.sql(f"INSERT OVERWRITE {tabela} SELECT * FROM pessoas_csv")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

### Parquet: UPDATE (esperado: erro)

**Esperado: erro (também na AWS).** Tabelas Parquet não aceitam `UPDATE`; cada tabela é executada em um `try` e a célula imprime a mensagem de erro em vez de interromper o notebook.

In [ ]:
for tabela in ("silver.tg_parquet", "silver.te_parquet"):
    try:
        spark.sql(f"UPDATE {tabela} SET valor = 999.0 WHERE id = 1")
        print(tabela, "-> UPDATE executado (inesperado)")
    except Exception as erro:
        print(tabela, "->", type(erro).__name__ + ":", str(erro).splitlines()[0][:300])

## 9. CSV

O CSV usa o nome simples (`silver.<tabela>`) no catálogo `glue`. Nesse formato o Spark SQL aceita `INSERT INTO` e `INSERT OVERWRITE`, mas **não** `UPDATE` nem `DELETE`.

### CSV: tabela gerenciada (sem `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS silver.tg_csv")
limpar("s3://notislake/silver/tg_csv/")
spark.sql("""
CREATE TABLE silver.tg_csv
USING csv
OPTIONS (header 'true')
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.tg_csv").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.tg_csv ORDER BY id").show()

### CSV: tabela externa (com `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS silver.te_csv")
limpar("s3://notislake/silver/te_csv/")
spark.sql("""
CREATE TABLE silver.te_csv
USING csv
OPTIONS (header 'true')
LOCATION 's3://notislake/silver/te_csv/'
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.te_csv").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.te_csv ORDER BY id").show()

### CSV: INSERT INTO e INSERT OVERWRITE

`INSERT INTO` acrescenta o `id` 6 (a tabela passa a ter 6 linhas); `INSERT OVERWRITE` troca todo o conteúdo pelo CSV de origem (volta para **5** linhas).

In [ ]:
for tabela in ("silver.tg_csv", "silver.te_csv"):
    spark.sql(f"INSERT INTO {tabela} VALUES (6, 'Fabio', 'Dados', 600.0, '2026-01-06')")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

In [ ]:
for tabela in ("silver.tg_csv", "silver.te_csv"):
    spark.sql(f"INSERT OVERWRITE {tabela} SELECT * FROM pessoas_csv")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

### CSV: UPDATE (esperado: erro)

**Esperado: erro (também na AWS).** Tabelas CSV não aceitam `UPDATE`; cada tabela é executada em um `try` e a célula imprime a mensagem de erro em vez de interromper o notebook.

In [ ]:
for tabela in ("silver.tg_csv", "silver.te_csv"):
    try:
        spark.sql(f"UPDATE {tabela} SET valor = 999.0 WHERE id = 1")
        print(tabela, "-> UPDATE executado (inesperado)")
    except Exception as erro:
        print(tabela, "->", type(erro).__name__ + ":", str(erro).splitlines()[0][:300])

## 10. ORC

O ORC usa o nome simples (`silver.<tabela>`) no catálogo `glue`. Nesse formato o Spark SQL aceita `INSERT INTO` e `INSERT OVERWRITE`, mas **não** `UPDATE` nem `DELETE`.

### ORC: tabela gerenciada (sem `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS silver.tg_orc")
limpar("s3://notislake/silver/tg_orc/")
spark.sql("""
CREATE TABLE silver.tg_orc
USING orc
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.tg_orc").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.tg_orc ORDER BY id").show()

### ORC: tabela externa (com `LOCATION`)

In [ ]:
spark.sql("DROP TABLE IF EXISTS silver.te_orc")
limpar("s3://notislake/silver/te_orc/")
spark.sql("""
CREATE TABLE silver.te_orc
USING orc
LOCATION 's3://notislake/silver/te_orc/'
AS SELECT * FROM pessoas_csv
""")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.te_orc").show(100, truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.te_orc ORDER BY id").show()

### ORC: INSERT INTO e INSERT OVERWRITE

`INSERT INTO` acrescenta o `id` 6 (a tabela passa a ter 6 linhas); `INSERT OVERWRITE` troca todo o conteúdo pelo CSV de origem (volta para **5** linhas).

In [ ]:
for tabela in ("silver.tg_orc", "silver.te_orc"):
    spark.sql(f"INSERT INTO {tabela} VALUES (6, 'Fabio', 'Dados', 600.0, '2026-01-06')")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

In [ ]:
for tabela in ("silver.tg_orc", "silver.te_orc"):
    spark.sql(f"INSERT OVERWRITE {tabela} SELECT * FROM pessoas_csv")
    spark.sql(f"SELECT id, nome, area, valor, dt FROM {tabela} ORDER BY id").show()

### ORC: UPDATE (esperado: erro)

**Esperado: erro (também na AWS).** Tabelas ORC não aceitam `UPDATE`; cada tabela é executada em um `try` e a célula imprime a mensagem de erro em vez de interromper o notebook.

In [ ]:
for tabela in ("silver.tg_orc", "silver.te_orc"):
    try:
        spark.sql(f"UPDATE {tabela} SET valor = 999.0 WHERE id = 1")
        print(tabela, "-> UPDATE executado (inesperado)")
    except Exception as erro:
        print(tabela, "->", type(erro).__name__ + ":", str(erro).splitlines()[0][:300])

# Parte B. DynamicFrame

O DynamicFrame lê e grava **arquivos** (aqui Parquet) e o catálogo só entra para dizer onde ficam e qual o schema. Por isso ele não tem o conceito de gerenciada/externa: o que ele cria é sempre uma tabela **EXTERNAL**. Pela documentação do Glue, o DynamicFrame também **não** lê nem grava tabelas Delta Lake, Hudi e Iceberg; para esses formatos usamos o Spark SQL da Parte A.

## 11. Criar tabela com getSink

`getSink` grava os arquivos em `s3://notislake/silver/te_dyf_sink/` e, com `enableUpdateCatalog=True` e `updateBehavior="UPDATE_IN_DATABASE"`, cria a tabela `silver.te_dyf_sink` no catálogo e registra as partições (`area`). A tabela aparece como EXTERNAL.

In [ ]:
destino_sink = "s3://notislake/silver/te_dyf_sink/"
pessoas_dyf = DynamicFrame.fromDF(df, glueContext, "pessoas")

spark.sql("DROP TABLE IF EXISTS silver.te_dyf_sink")
limpar(destino_sink)

sink = glueContext.getSink(
    connection_type="s3",
    path=destino_sink,
    enableUpdateCatalog=True,
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["area"],
)
sink.setCatalogInfo(catalogDatabase="silver", catalogTableName="te_dyf_sink")
sink.setFormat("glueparquet")
sink.writeFrame(pessoas_dyf)

spark.catalog.refreshTable("silver.te_dyf_sink")

In [ ]:
spark.sql("DESCRIBE TABLE EXTENDED silver.te_dyf_sink").show(100, truncate=False)
spark.sql("SHOW PARTITIONS silver.te_dyf_sink").show(truncate=False)
spark.sql("SELECT id, nome, area, valor, dt FROM silver.te_dyf_sink ORDER BY id").show()

## 12. Ler com from_catalog

`from_catalog` lê a tabela pelo nome, sem informar caminho nem schema. Lemos `tg_parquet` (gerenciada) e `te_parquet` (externa), criadas na Parte A; ambas têm **5** linhas.

In [ ]:
tg_parquet_dyf = glueContext.create_dynamic_frame.from_catalog(database="silver", table_name="tg_parquet")
tg_parquet_dyf.toDF().orderBy("id").show()

In [ ]:
te_parquet_dyf = glueContext.create_dynamic_frame.from_catalog(database="silver", table_name="te_parquet")
te_parquet_dyf.toDF().orderBy("id").show()

## 13. Acrescentar com write_dynamic_frame.from_catalog

`write_dynamic_frame.from_catalog` grava no `location` e no formato já registrados na tabela `te_parquet` e acrescenta 1 linha (`id` 7); a tabela passa a ter **6** linhas. Como o Spark guarda em cache a lista de arquivos da tabela, `spark.catalog.refreshTable` a atualiza. Para voltar a 5 linhas, reexecute a seção do Parquet da Parte A.

In [ ]:
nova = spark.createDataFrame([(7, "Gabi", "Dados", 700.0, "2026-01-07")], schema)
nova_dyf = DynamicFrame.fromDF(nova, glueContext, "nova")

glueContext.write_dynamic_frame.from_catalog(
    frame=nova_dyf,
    database="silver",
    table_name="te_parquet",
)

spark.catalog.refreshTable("silver.te_parquet")
spark.sql("SELECT id, nome, area, valor, dt FROM silver.te_parquet ORDER BY id").show()

## 14. Resumo

Para cada tabela lemos o tipo (`TableType`) e o local (`Location`) **direto do catálogo `glue`** (API `get_table`, que não depende do rótulo do Spark) e contamos as linhas. Cada leitura fica em um `try` próprio: se uma etapa anterior falhou, o resumo mostra `ERRO` em vez de interromper o notebook. O esperado é **5** linhas em todas as tabelas, **6** em `te_parquet` (linha acrescentada na Parte B), `tg_*` e `te_*` do Iceberg como EXTERNAL e as demais `tg_*` como MANAGED.

In [ ]:
import boto3

tabelas = [
    ("glue_catalog.silver.tg_iceberg", "iceberg"),
    ("glue_catalog.silver.te_iceberg", "iceberg"),
    ("silver.tg_delta", "delta"),
    ("silver.te_delta", "delta"),
    ("silver.tg_hudi", "hudi"),
    ("silver.te_hudi", "hudi"),
    ("silver.tg_parquet", "parquet"),
    ("silver.te_parquet", "parquet"),
    ("silver.tg_csv", "csv"),
    ("silver.te_csv", "csv"),
    ("silver.tg_orc", "orc"),
    ("silver.te_orc", "orc"),
    ("silver.te_dyf_sink", "glueparquet (DynamicFrame)"),
]

glue = boto3.client("glue")
resumo = []
for tabela, formato in tabelas:
    try:
        banco, nome = tabela.split(".")[-2:]
        registro = glue.get_table(DatabaseName=banco, Name=nome)["Table"]
        tipo = registro["TableType"].replace("_TABLE", "")
        local = registro["StorageDescriptor"]["Location"]
    except Exception as erro:
        tipo, local = "ERRO: " + type(erro).__name__ + ": " + " ".join(str(erro).split())[:80], "-"
    try:
        linhas = str(spark.table(tabela).count())
    except Exception as erro:
        linhas = "ERRO: " + type(erro).__name__ + ": " + " ".join(str(erro).split())[:80]
    resumo.append((tabela, formato, tipo, local, linhas))

spark.createDataFrame(
    resumo, "tabela STRING, formato STRING, tipo STRING, location STRING, linhas STRING"
).show(truncate=False)